# Enrich existing Wikidata benchmark gold answers

Этот ноутбук **не генерирует новые вопросы**. Он берёт уже собранный `dataset_main.jsonl`, читает сохранённый `sparql_query` у каждого примера и пытается дособрать полный список gold-ответов, которые были обрезаны старым `LIMIT`.

Идея:

1. прочитать текущий JSONL и проверить его структуру;
2. для каждого примера вывести из сохранённого SPARQL переменную ответа (`?item`, `?film`, `?book`, ...);
3. убрать внешний `LIMIT/OFFSET`;
4. сделать быстрый QID-only запрос без label-части;
5. отдельно подтянуть labels через Wikidata API батчами;
6. сохранить расширенный датасет с полями `*_full`, при этом оригинальный gold тоже сохранить.

Главная защита от зависаний: если запрос слишком широкий, ноутбук **не сохраняет частичный gold как будто он полный**, а помечает пример статусом `too_many_candidates`.

In [12]:
# ============================================================
# 0. CONFIG
# ============================================================
from pathlib import Path

# Входные файлы из твоего основного ноутбука.
INPUT_MAIN_PATH = Path("out_wikidata_benchmark/dataset_main.jsonl")
INPUT_ZERO_PATH = Path("out_wikidata_benchmark/dataset_zero.jsonl")

# Новые выходные файлы. Старый dataset_main.jsonl не перезаписывается.
OUTPUT_MAIN_PATH = Path("out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl")
OUTPUT_ZERO_PATH = Path("out_wikidata_benchmark/dataset_zero.full_gold_enriched.jsonl")

# Что обрабатывать.
ENRICH_MAIN = True
ENRICH_ZERO = False        # Обычно сначала лучше чинить только MAIN. ZERO можно включить потом для проверки провокаций.

# None = все домены. Для быстрой проверки можно поставить {"cinema"}.
DOMAIN_FILTER = None       # пример: {"cinema"}
COMPLEXITY_FILTER = None   # пример: {"L3", "L4", "L5"}
MAX_RECORDS = None         # пример: 20 для smoke-test

# Если True, основной gold_answer_qids/gold_answer_labels_ru будет заменён на full-gold.
# Оригинальный gold всё равно сохранится в gold_answer_qids_original/gold_answer_labels_ru_original.
REPLACE_PRIMARY_GOLD_FIELDS = True

# Resume: если output уже есть, обработанные id будут пропущены.
RESUME = True
FSYNC_EACH_LINE = False

# Ограничение на полный gold. Это не "обрезание": если кандидатов больше лимита,
# пример помечается too_many_candidates и основной gold не заменяется.
# Если хочешь вообще без такого ограничения, поставь None, но это может надолго подвесить WDQS на широких запросах.
MAX_QIDS_PER_EXAMPLE_BY_DOMAIN = {
    "cinema": 700,
    "books": 700,
    "dishes": 400,
    "countries": 600,
    "airports": 900,
    "universities": 900,
    "cars": 900,
    "smartphones": 900,
    "software": 900,
    "videogames": 900,
    "default": 800,
}

# Скорость/надёжность WDQS.
WDQS_ENDPOINT = "https://query.wikidata.org/sparql"
WIKI_API = "https://www.wikidata.org/w/api.php"
USER_AGENT = "YandexGPT-reversal-curse-benchmark-gold-enricher (contact: salam121asd@gmail.com)"

WDQS_TIMEOUT_SECONDS = 18
WDQS_MAX_RETRIES = 2
WDQS_MIN_DELAY_SECONDS = 0.8
WIKI_API_TIMEOUT_SECONDS = 20

# Быстрый режим: переписать сохранённый SELECT в SELECT DISTINCT ?answerVar и убрать label-строки.
# Если быстрый режим дал подозрительный результат, ноут сам fallback-ится на wrapped original query.
USE_FAST_QID_ONLY_REWRITE = True

# Политика labels.
# Для большинства RU-запросов лучше требовать русскую label. Для некоторых доменов нормальны латинские названия.
REQUIRE_RU_LABEL_BY_DEFAULT = True
ALLOW_NON_RU_FALLBACK_DOMAINS = {
    "airports", "cars", "universities", "smartphones", "software", "videogames"
}

# Кэш. Можно безопасно переиспользовать между запусками.
CACHE_DIR = Path("out_wikidata_benchmark/cache_gold_enrichment")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RUN_ENRICHMENT_NOW = True

print("INPUT_MAIN_PATH =", INPUT_MAIN_PATH)
print("OUTPUT_MAIN_PATH =", OUTPUT_MAIN_PATH)
print("DOMAIN_FILTER =", DOMAIN_FILTER)

INPUT_MAIN_PATH = out_wikidata_benchmark/dataset_main.jsonl
OUTPUT_MAIN_PATH = out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl
DOMAIN_FILTER = None


In [2]:
# ============================================================
# 1. Imports + JSONL helpers
# ============================================================
import os
import re
import json
import time
import math
import hashlib
import random
import traceback
from dataclasses import dataclass
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple

import requests
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

QID_RE = re.compile(r"^Q\d+$")


def read_jsonl(path: Path, limit: Optional[int] = None) -> List[dict]:
    rows = []
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and len(rows) >= limit:
                break
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[WARN] Bad JSONL line #{i+1}; loaded {len(rows)} rows before it. Error: {e}")
                break
    return rows


def iter_jsonl(path: Path) -> Iterator[dict]:
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                print(f"[WARN] Bad JSONL line #{i+1}; stop reading. Error: {e}")
                return


def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    n = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                n += 1
    return n


def append_jsonl(path: Path, obj: dict, fsync: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        if fsync:
            f.flush()
            os.fsync(f.fileno())


def write_jsonl(path: Path, rows: Iterable[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for obj in rows:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def safe_len_list(x: Any) -> int:
    return len(x) if isinstance(x, list) else 0


def uri_to_qid(value: str) -> Optional[str]:
    if not isinstance(value, str):
        return None
    if QID_RE.match(value):
        return value
    m = re.search(r"/(Q\d+)$", value)
    return m.group(1) if m else None


def unique_keep_order(items: Iterable[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Проверка структуры текущего датасета

Эта ячейка специально оставлена в ноутбуке: перед enrichment она показывает реальные поля, распределение по доменам/сложностям, наличие SPARQL, старые размеры gold и признаки обрезания.

In [3]:
# ============================================================
# 2. Dataset structure inspection
# ============================================================

def extract_select_variables(query: str) -> List[str]:
    if not isinstance(query, str) or not query.strip():
        return []
    q = re.sub(r"(?m)#.*$", "", query)
    m = re.search(r"(?is)\bSELECT\b\s+(?:DISTINCT\s+|REDUCED\s+)?(.*?)\bWHERE\b", q)
    if not m:
        return []
    clause = m.group(1)
    vars_ = re.findall(r"\?([A-Za-z_][A-Za-z0-9_]*)", clause)
    return list(dict.fromkeys(vars_))


def has_outer_limit(query: str) -> bool:
    if not isinstance(query, str):
        return False
    q = re.sub(r"(?m)#.*$", "", query).strip()
    return bool(re.search(r"(?is)\bLIMIT\s+\d+\s*(?:OFFSET\s+\d+\s*)?$", q))


def inspect_dataset(path: Path, sample_n: int = 5) -> pd.DataFrame:
    rows = read_jsonl(path)
    print(f"Loaded {len(rows)} records from {path}")
    if not rows:
        return pd.DataFrame()

    all_keys = sorted({k for r in rows for k in r.keys()})
    print("\nTop-level fields:")
    print(all_keys)

    df = pd.DataFrame(rows)
    for col in ["id", "domain", "complexity", "requested_count", "gold_answer_qids", "gold_answer_labels_ru", "sparql_query", "gold_truncated"]:
        if col not in df.columns:
            df[col] = None

    df["gold_old_count"] = df["gold_answer_qids"].apply(safe_len_list)
    df["labels_old_count"] = df["gold_answer_labels_ru"].apply(safe_len_list)
    df["has_sparql"] = df["sparql_query"].apply(lambda x: isinstance(x, str) and bool(x.strip()))
    df["has_outer_limit"] = df["sparql_query"].apply(has_outer_limit)
    df["select_vars"] = df["sparql_query"].apply(extract_select_variables)

    print("\nDomain/complexity counts:")
    display(df.groupby(["domain", "complexity"], dropna=False).size().to_frame("n"))

    print("\nGold size stats by domain/complexity:")
    display(
        df.groupby(["domain", "complexity"], dropna=False)
          .agg(
              n=("id", "count"),
              mean_gold=("gold_old_count", "mean"),
              median_gold=("gold_old_count", "median"),
              max_gold=("gold_old_count", "max"),
              p_has_limit=("has_outer_limit", "mean"),
              p_gold_truncated=("gold_truncated", lambda s: pd.Series(s).fillna(False).astype(bool).mean()),
          )
          .reset_index()
    )

    mismatch = df[df["gold_old_count"] != df["labels_old_count"]]
    if len(mismatch):
        print(f"\n[WARN] qids/labels count mismatch: {len(mismatch)} rows")
        display(mismatch[["id", "domain", "complexity", "gold_old_count", "labels_old_count"]].head(10))

    print("\nMost common SELECT variable sets:")
    vc = Counter(tuple(v) for v in df["select_vars"].tolist())
    for vars_, cnt in vc.most_common(12):
        print(f"{cnt:4d}  {vars_}")

    print("\nSample records:")
    show_cols = ["id", "domain", "complexity", "query_text_ru", "requested_count", "gold_old_count", "gold_truncated", "has_outer_limit", "select_vars"]
    display(df[show_cols].head(sample_n))

    return df

if INPUT_MAIN_PATH.exists():
    df_input_main = inspect_dataset(INPUT_MAIN_PATH, sample_n=5)
else:
    print(f"[INFO] {INPUT_MAIN_PATH} пока не найден. Положи dataset_main.jsonl рядом с ноутбуком или поправь INPUT_MAIN_PATH.")

[WARN] Bad JSONL line #133; loaded 132 rows before it. Error: Extra data: line 1 column 1300 (char 1299)
Loaded 132 records from out_wikidata_benchmark/dataset_main.jsonl

Top-level fields:
['ask_validator_sparql', 'complexity', 'constraints', 'created_at', 'domain', 'gold_answer_labels_ru', 'gold_answer_qids', 'gold_truncated', 'id', 'is_advanced', 'query_text_ru', 'requested_count', 'sparql_query', 'template_family', 'template_id']

Domain/complexity counts:


n
domain    complexity    
cinema    L1          41
          L2          30
          L3           6
          L4          27
          L5          18
countries L2          10


Gold size stats by domain/complexity:


,domain,complexity,n,mean_gold,median_gold,max_gold,p_has_limit,p_gold_truncated
0,cinema,L1,41,17.658537,25.0,25,1.0,0.512195
1,cinema,L2,30,6.766667,6.0,19,1.0,0.000000
2,cinema,L3,6,7.166667,6.5,10,1.0,0.000000
3,cinema,L4,27,21.444444,25.0,25,1.0,0.592593
4,cinema,L5,18,13.111111,8.0,25,1.0,0.333333
5,countries,L2,10,3.600000,3.5,5,1.0,0.000000



Most common SELECT variable sets:
 122  ('item', 'itemLabel')
  10  ('c', 'cLabel')

Sample records:


,id,domain,complexity,query_text_ru,requested_count,gold_old_count,gold_truncated,has_outer_limit,select_vars
0,cinema_l1_0000,cinema,L1,"Назови 5 фильмов, жанра «криминальный фильм», ...",5,25,True,True,"[item, itemLabel]"
1,cinema_l1_0002,cinema,L1,"Назови 5 фильмов, жанра «драма», в период 2018...",5,8,False,True,"[item, itemLabel]"
2,cinema_l1_0004,cinema,L1,"Назови 5 фильмов, жанра «криминальный фильм», ...",5,25,True,True,"[item, itemLabel]"
3,cinema_l1_0006,cinema,L1,"Назови 5 фильмов, жанра «драма», в период 2005...",5,18,False,True,"[item, itemLabel]"
4,cinema_l1_0008,cinema,L1,"Назови 5 фильмов, жанра «криминальный фильм», ...",5,25,True,True,"[item, itemLabel]"


In [4]:
# ============================================================
# 3. Disk cache + Wikidata clients
# ============================================================

class JsonDiskCache:
    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def _path(self, namespace: str, key: str) -> Path:
        d = self.root / namespace
        d.mkdir(parents=True, exist_ok=True)
        return d / f"{key}.json"

    @staticmethod
    def make_key(text: str) -> str:
        return hashlib.sha1(text.encode("utf-8")).hexdigest()

    def get(self, namespace: str, text: str) -> Optional[dict]:
        path = self._path(namespace, self.make_key(text))
        if not path.exists():
            return None
        try:
            with path.open("r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            try:
                path.rename(path.with_suffix(path.suffix + ".bad"))
            except Exception:
                pass
            return None

    def set(self, namespace: str, text: str, value: dict) -> None:
        path = self._path(namespace, self.make_key(text))
        tmp = path.with_suffix(".tmp")
        with tmp.open("w", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False)
        tmp.replace(path)


cache = JsonDiskCache(CACHE_DIR)


class WDQSClient:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": USER_AGENT,
            "Accept": "application/sparql-results+json",
        })
        self.last_request_ts = 0.0

    def _wait_rate_limit(self):
        delta = time.time() - self.last_request_ts
        if delta < WDQS_MIN_DELAY_SECONDS:
            time.sleep(WDQS_MIN_DELAY_SECONDS - delta)

    def select(self, query: str, use_cache: bool = True, timeout: Optional[int] = None) -> dict:
        q = query.strip()
        if use_cache:
            got = cache.get("wdqs_select", q)
            if got is not None:
                return got

        last_exc = None
        timeout = timeout or WDQS_TIMEOUT_SECONDS
        for attempt in range(WDQS_MAX_RETRIES + 1):
            self._wait_rate_limit()
            resp = None
            try:
                resp = self.session.post(
                    WDQS_ENDPOINT,
                    data={"query": q, "format": "json"},
                    timeout=timeout,
                )
                self.last_request_ts = time.time()
                if resp.status_code in (429, 500, 502, 503, 504):
                    raise RuntimeError(f"WDQS transient HTTP {resp.status_code}: {resp.text[:300]}")
                if resp.status_code >= 400:
                    raise RuntimeError(f"WDQS HTTP {resp.status_code}: {resp.text[:800]}")
                data = resp.json()
                if use_cache:
                    cache.set("wdqs_select", q, data)
                return data
            except Exception as e:
                last_exc = e
                if attempt >= WDQS_MAX_RETRIES:
                    break
                sleep_s = min(12.0, 1.5 * (2 ** attempt)) + random.random()
                time.sleep(sleep_s)
        raise RuntimeError(f"WDQS failed after retries: {last_exc}")


wdqs = WDQSClient()

In [5]:
# ============================================================
# 4. SPARQL rewrite helpers
# ============================================================

PREFERRED_ANSWER_VARS = [
    "item", "film", "movie", "work", "book", "game", "album", "person", "scientist",
    "physicist", "mathematician", "dish", "country", "place", "airport", "university",
    "car", "software", "smartphone", "painting", "museum", "club", "spacecraft", "val"
]

LABEL_VAR_SUFFIXES = (
    "label", "labelru", "labelen", "title", "titleru", "name", "nameru", "description"
)

NON_ANSWER_VAR_NAMES = {
    "date", "year", "genre", "country", "director", "actor", "linkActor", "author", "publisher",
    "score", "rating", "votes", "runtime", "series", "prev", "next", "sl", "seed", "class",
    "female", "male", "valLabelRu", "valLabel", "itemLabel", "itemLabelRu"
}


def strip_sparql_comments(q: str) -> str:
    # В наших запросах комментарии не нужны для исполнения. Упрощает удаление внешнего LIMIT.
    return re.sub(r"(?m)#.*$", "", q or "")


def strip_outer_limit_offset(q: str) -> str:
    q = strip_sparql_comments(q).strip().rstrip(";").strip()
    # Удаляем только финальные LIMIT/OFFSET. Внутренние LIMIT в подзапросах не трогаем.
    for _ in range(4):
        old = q
        q = re.sub(r"(?is)\s+OFFSET\s+\d+\s*$", "", q).strip()
        q = re.sub(r"(?is)\s+LIMIT\s+\d+\s*$", "", q).strip()
        if q == old:
            break
    return q


def strip_outer_order_by(q: str) -> str:
    # Если ORDER BY стоит в самом конце и ссылается на label-переменную, после rewrite он может ломать запрос.
    # Внутренние ORDER BY внутри подзапросов не трогаются обычным regex, если после них не конец всего запроса.
    q = q.strip()
    q = re.sub(r"(?is)\s+ORDER\s+BY\s+[^{}]*$", "", q).strip()
    return q


def infer_answer_var(ex: dict) -> Optional[str]:
    q = ex.get("sparql_query") or ""
    vars_ = extract_select_variables(q)
    if not vars_:
        # ASK validator обычно содержит BIND(wd:{ITEM} AS ?item)
        ask = ex.get("ask_validator_sparql") or ""
        m = re.search(r"(?is)BIND\s*\(\s*wd:\{ITEM\}\s+AS\s+\?([A-Za-z_][A-Za-z0-9_]*)\s*\)", ask)
        return m.group(1) if m else None

    lower_to_orig = {v.lower(): v for v in vars_}
    for pref in PREFERRED_ANSWER_VARS:
        if pref.lower() in lower_to_orig:
            return lower_to_orig[pref.lower()]

    candidates = []
    for v in vars_:
        vl = v.lower()
        if any(vl.endswith(suf) for suf in LABEL_VAR_SUFFIXES):
            continue
        if vl in {x.lower() for x in NON_ANSWER_VAR_NAMES}:
            continue
        candidates.append(v)
    if candidates:
        return candidates[0]
    return vars_[0]


def remove_service_wikibase_label_blocks(q: str) -> str:
    """Удаляет SERVICE wikibase:label { ... } с балансом фигурных скобок."""
    s = q
    pattern = re.compile(r"(?is)SERVICE\s+wikibase:label\s*\{")
    pos = 0
    out = []
    while True:
        m = pattern.search(s, pos)
        if not m:
            out.append(s[pos:])
            break
        out.append(s[pos:m.start()])
        depth = 0
        i = m.end() - 1  # на открывающей скобке
        end = None
        while i < len(s):
            if s[i] == "{":
                depth += 1
            elif s[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
            i += 1
        if end is None:
            # если что-то странное, лучше не ломать запрос
            out.append(s[m.start():])
            break
        pos = end
    return "".join(out)


def remove_obvious_label_lines(q: str) -> str:
    """Убирает output-only label строки. Семантические ограничения по QID остаются."""
    lines = q.splitlines()
    kept = []
    for line in lines:
        l = line.strip()
        low = l.lower()
        # rdfs:label для переменной *Label* почти всегда нужен только для вывода.
        if "rdfs:label" in low and re.search(r"\?[A-Za-z_][A-Za-z0-9_]*(Label|LabelRu|LabelEn)\b", l):
            continue
        # FILTER(LANG(?...Label...)) после удаления label-строки становится мусором.
        if "filter" in low and "lang(" in low and re.search(r"\?[A-Za-z_][A-Za-z0-9_]*(Label|LabelRu|LabelEn)\b", l):
            continue
        # schema:name/schema:description output-only варианты.
        if ("schema:name" in low or "schema:description" in low) and re.search(r"\?[A-Za-z_][A-Za-z0-9_]*(Label|LabelRu|Name|Title)\b", l):
            continue
        kept.append(line)
    return "\n".join(kept)


def replace_select_with_qid_only(q: str, answer_var: str) -> str:
    # Заменяем SELECT ... WHERE на SELECT DISTINCT ?answer_var WHERE.
    # Работает и при PREFIX строках до SELECT.
    m = re.search(r"(?is)\bSELECT\b\s+(?:DISTINCT\s+|REDUCED\s+)?(.*?)\bWHERE\b\s*\{", q)
    if not m:
        raise ValueError("Cannot find SELECT ... WHERE in SPARQL")
    return q[:m.start()] + f"SELECT DISTINCT ?{answer_var} WHERE {{" + q[m.end():]


def build_fast_qid_query(original_query: str, answer_var: str) -> str:
    q = strip_outer_limit_offset(original_query)
    q = strip_outer_order_by(q)
    q = remove_service_wikibase_label_blocks(q)
    q = remove_obvious_label_lines(q)
    q = replace_select_with_qid_only(q, answer_var)
    return q.strip()


def build_wrapped_original_qid_query(original_query: str, answer_var: str) -> str:
    # Самый безопасный fallback: сохранить оригинальную семантику запроса как subquery,
    # но снять внешний LIMIT и выбрать только переменную ответа.
    inner = strip_outer_limit_offset(original_query)
    inner = strip_outer_order_by(inner)
    return f"""
SELECT DISTINCT ?{answer_var} WHERE {{
  {{
{inner}
  }}
}}
""".strip()


def with_limit(query_without_limit: str, limit: int) -> str:
    q = strip_outer_limit_offset(query_without_limit)
    return f"{q}\nLIMIT {int(limit)}"


def qids_from_wdqs_result(data: dict, answer_var: str) -> List[str]:
    rows = data.get("results", {}).get("bindings", [])
    qids = []
    for row in rows:
        # Сначала пробуем ожидаемую переменную.
        if answer_var in row:
            qid = uri_to_qid(row[answer_var].get("value"))
            if qid:
                qids.append(qid)
                continue
        # Fallback: первая URI-переменная, похожая на QID.
        for v in row.values():
            qid = uri_to_qid(v.get("value")) if isinstance(v, dict) else None
            if qid:
                qids.append(qid)
                break
    return unique_keep_order(qids)


def quick_sparql_debug(ex: dict) -> dict:
    answer_var = infer_answer_var(ex)
    q = ex.get("sparql_query") or ""
    return {
        "id": ex.get("id"),
        "domain": ex.get("domain"),
        "complexity": ex.get("complexity"),
        "answer_var": answer_var,
        "select_vars": extract_select_variables(q),
        "has_outer_limit": has_outer_limit(q),
        "old_gold_count": safe_len_list(ex.get("gold_answer_qids")),
    }

# Небольшой dry check на первых строках, без похода в Wikidata.
if INPUT_MAIN_PATH.exists():
    sample_rows = read_jsonl(INPUT_MAIN_PATH, limit=5)
    display(pd.DataFrame([quick_sparql_debug(x) for x in sample_rows]))

,id,domain,complexity,answer_var,select_vars,has_outer_limit,old_gold_count
0,cinema_l1_0000,cinema,L1,item,"[item, itemLabel]",True,25
1,cinema_l1_0002,cinema,L1,item,"[item, itemLabel]",True,8
2,cinema_l1_0004,cinema,L1,item,"[item, itemLabel]",True,25
3,cinema_l1_0006,cinema,L1,item,"[item, itemLabel]",True,18
4,cinema_l1_0008,cinema,L1,item,"[item, itemLabel]",True,25


In [6]:
# ============================================================
# 5. Wikidata label fetcher
# ============================================================

class WikidataEntityClient:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": USER_AGENT})

    def fetch_entities(self, qids: List[str], use_cache: bool = True) -> Dict[str, dict]:
        qids = [q for q in unique_keep_order(qids) if QID_RE.match(q)]
        out = {}
        missing = []
        for qid in qids:
            got = cache.get("entity", qid) if use_cache else None
            if got is not None:
                out[qid] = got
            else:
                missing.append(qid)

        # Wikidata API обычно позволяет до 50 ids за запрос без bot-rights.
        for i in range(0, len(missing), 50):
            batch = missing[i:i+50]
            params = {
                "action": "wbgetentities",
                "format": "json",
                "ids": "|".join(batch),
                "props": "labels|aliases|sitelinks",
                "languages": "ru|en",
                "languagefallback": 1,
            }
            last_exc = None
            for attempt in range(3):
                try:
                    resp = self.session.get(WIKI_API, params=params, timeout=WIKI_API_TIMEOUT_SECONDS)
                    if resp.status_code in (429, 500, 502, 503, 504):
                        raise RuntimeError(f"Wikidata API transient HTTP {resp.status_code}")
                    resp.raise_for_status()
                    data = resp.json()
                    entities = data.get("entities", {})
                    for qid, ent in entities.items():
                        out[qid] = ent
                        cache.set("entity", qid, ent)
                    break
                except Exception as e:
                    last_exc = e
                    if attempt == 2:
                        print(f"[WARN] label batch failed {batch[:3]}...: {last_exc}")
                    time.sleep(1.0 + attempt + random.random())
        return out


entities_client = WikidataEntityClient()


def domain_allows_label_fallback(domain: str) -> bool:
    return str(domain) in ALLOW_NON_RU_FALLBACK_DOMAINS


def choose_entity_label(ent: dict, domain: str) -> Tuple[Optional[str], str]:
    """Returns (label, source). source: ru_label | ru_alias | en_label | fallback | none."""
    if not isinstance(ent, dict) or ent.get("missing"):
        return None, "none"

    labels = ent.get("labels") or {}
    aliases = ent.get("aliases") or {}

    ru = labels.get("ru", {}).get("value")
    if ru:
        return ru.strip(), "ru_label"

    ru_aliases = aliases.get("ru") or []
    for a in ru_aliases:
        val = a.get("value")
        if val:
            return val.strip(), "ru_alias"

    if REQUIRE_RU_LABEL_BY_DEFAULT and not domain_allows_label_fallback(domain):
        return None, "no_ru_label"

    en = labels.get("en", {}).get("value")
    if en:
        return en.strip(), "en_label"

    # Последний fallback: любой label из Wikidata, но только для разрешённых доменов.
    if domain_allows_label_fallback(domain):
        for obj in labels.values():
            val = obj.get("value")
            if val:
                return val.strip(), "any_label"

    return None, "none"


def fetch_labels_aligned(qids: List[str], domain: str) -> Tuple[List[str], List[str], dict]:
    """Возвращает только qids с usable label, labels aligned, и meta."""
    ents = entities_client.fetch_entities(qids)
    out_qids, out_labels = [], []
    source_counter = Counter()
    dropped_no_label = []

    for qid in qids:
        label, source = choose_entity_label(ents.get(qid, {}), domain)
        source_counter[source] += 1
        if label:
            out_qids.append(qid)
            out_labels.append(label)
        else:
            dropped_no_label.append(qid)

    meta = {
        "label_sources": dict(source_counter),
        "dropped_no_usable_label_count": len(dropped_no_label),
        "dropped_no_usable_label_qids_preview": dropped_no_label[:20],
    }
    return out_qids, out_labels, meta

In [7]:
# ============================================================
# 6. Gold collection for one example
# ============================================================

@dataclass
class GoldCollectResult:
    status: str
    qids_raw: List[str]
    answer_var: Optional[str]
    method: Optional[str]
    meta: Dict[str, Any]


def max_qids_for_domain(domain: str) -> Optional[int]:
    return MAX_QIDS_PER_EXAMPLE_BY_DOMAIN.get(str(domain), MAX_QIDS_PER_EXAMPLE_BY_DOMAIN.get("default"))


def try_collect_with_base_query(
    base_query: str,
    answer_var: str,
    limit: Optional[int],
    timeout: int = WDQS_TIMEOUT_SECONDS,
) -> Tuple[str, List[str], dict]:
    """
    Возвращает status, qids, meta.
    Если limit не None, запрашивает limit+1, чтобы понять too_many без частичного сохранения.
    """
    if limit is None:
        # Без лимита опасно, но поддержано. Для 1k датасета лучше не использовать.
        query = strip_outer_limit_offset(base_query)
        data = wdqs.select(query, timeout=timeout)
        qids = qids_from_wdqs_result(data, answer_var)
        return "ok", qids, {"query_limit_used": None, "rows_returned": len(qids)}

    query_limit = int(limit) + 1
    query = with_limit(base_query, query_limit)
    data = wdqs.select(query, timeout=timeout)
    qids = qids_from_wdqs_result(data, answer_var)

    if len(qids) > limit:
        return "too_many_candidates", [], {
            "max_qids_allowed": limit,
            "query_limit_used": query_limit,
            "rows_returned": len(qids),
            "note": "query has more candidates than safety cap; full gold not saved as partial",
        }
    return "ok", qids, {"max_qids_allowed": limit, "query_limit_used": query_limit, "rows_returned": len(qids)}


def collect_full_gold_qids(ex: dict) -> GoldCollectResult:
    original_query = ex.get("sparql_query") or ""
    domain = str(ex.get("domain") or "")
    old_qids = [q for q in (ex.get("gold_answer_qids") or []) if isinstance(q, str) and QID_RE.match(q)]

    if not original_query.strip():
        return GoldCollectResult("skip_no_sparql", [], None, None, {})

    answer_var = infer_answer_var(ex)
    if not answer_var:
        return GoldCollectResult("skip_cannot_infer_answer_var", [], None, None, {"select_vars": extract_select_variables(original_query)})

    limit = max_qids_for_domain(domain)
    attempts = []

    # 1) Fast rewritten QID-only query.
    if USE_FAST_QID_ONLY_REWRITE:
        try:
            fast_query = build_fast_qid_query(original_query, answer_var)
            status, qids, meta = try_collect_with_base_query(fast_query, answer_var, limit)
            meta.update({"method": "fast_qid_rewrite"})
            attempts.append({"method": "fast_qid_rewrite", "status": status, **meta})

            if status == "ok":
                missing_old = [q for q in old_qids if q not in set(qids)]
                if not missing_old:
                    return GoldCollectResult(status, qids, answer_var, "fast_qid_rewrite", {"attempts": attempts})
                # Если быстрый rewrite изменил семантику, fallback на wrapped original.
                attempts[-1]["old_qids_missing_from_fast_preview"] = missing_old[:20]
            elif status == "too_many_candidates":
                # Too many в fast режиме обычно достаточно. Но если старый запрос был очень label-specific,
                # wrapped original может оказаться меньше. Поэтому fallback всё равно попробуем.
                pass
        except Exception as e:
            attempts.append({"method": "fast_qid_rewrite", "status": "error", "error": repr(e)[:1000]})

    # 2) Safe fallback: wrapped original query with original label constraints.
    try:
        wrapped_query = build_wrapped_original_qid_query(original_query, answer_var)
        status, qids, meta = try_collect_with_base_query(wrapped_query, answer_var, limit)
        meta.update({"method": "wrapped_original"})
        attempts.append({"method": "wrapped_original", "status": status, **meta})

        if status == "ok":
            missing_old = [q for q in old_qids if q not in set(qids)]
            if missing_old:
                meta["old_qids_missing_from_wrapped_preview"] = missing_old[:20]
                return GoldCollectResult("ok_but_original_not_subset", qids, answer_var, "wrapped_original", {"attempts": attempts})
        return GoldCollectResult(status, qids, answer_var, "wrapped_original", {"attempts": attempts})
    except Exception as e:
        attempts.append({"method": "wrapped_original", "status": "error", "error": repr(e)[:1500]})
        return GoldCollectResult("error", [], answer_var, None, {"attempts": attempts})


def enrich_one_example(ex: dict) -> dict:
    ex = dict(ex)  # не мутируем исходный объект
    domain = str(ex.get("domain") or "")
    old_qids = [q for q in (ex.get("gold_answer_qids") or []) if isinstance(q, str)]
    old_labels = list(ex.get("gold_answer_labels_ru") or [])

    # Сохраняем оригинал ровно один раз.
    ex.setdefault("gold_answer_qids_original", old_qids)
    ex.setdefault("gold_answer_labels_ru_original", old_labels)
    ex["gold_old_count"] = len(old_qids)

    result = collect_full_gold_qids(ex)
    ex["gold_enrichment_status"] = result.status
    ex["gold_enrichment_method"] = result.method
    ex["gold_enrichment_answer_var"] = result.answer_var
    ex["gold_enrichment_meta"] = result.meta

    if result.status not in {"ok", "ok_but_original_not_subset"}:
        ex["gold_full_count"] = None
        ex["gold_added_count"] = None
        ex["gold_missing_from_original_count"] = None
        return ex

    qids_raw = unique_keep_order(result.qids_raw)
    qids_labeled, labels, label_meta = fetch_labels_aligned(qids_raw, domain)

    old_set = set(old_qids)
    full_set = set(qids_labeled)
    added = [q for q in qids_labeled if q not in old_set]
    old_missing_from_full = [q for q in old_qids if q not in full_set]

    ex["gold_answer_qids_full_raw"] = qids_raw
    ex["gold_answer_qids_full"] = qids_labeled
    ex["gold_answer_labels_ru_full"] = labels
    ex["gold_full_count_raw"] = len(qids_raw)
    ex["gold_full_count"] = len(qids_labeled)
    ex["gold_added_count"] = len(added)
    ex["gold_added_qids_preview"] = added[:30]
    ex["gold_missing_from_original_count"] = len(old_missing_from_full)
    ex["gold_missing_from_original_qids_preview"] = old_missing_from_full[:30]
    ex["gold_label_meta"] = label_meta
    ex["gold_truncated"] = False
    ex["gold_was_enriched"] = True

    if REPLACE_PRIMARY_GOLD_FIELDS:
        ex["gold_answer_qids"] = qids_labeled
        ex["gold_answer_labels_ru"] = labels

    return ex

In [35]:
# ============================================================
# 7. Full dataset enrichment with resume
# ============================================================

SUCCESS_STATUSES = {"ok", "ok_but_original_not_subset"}
SKIP_STATUSES = {"skip_no_sparql", "skip_cannot_infer_answer_var"}


def record_passes_filters(ex: dict) -> bool:
    if DOMAIN_FILTER is not None and ex.get("domain") not in DOMAIN_FILTER:
        return False
    if COMPLEXITY_FILTER is not None and ex.get("complexity") not in COMPLEXITY_FILTER:
        return False
    return True


def load_processed_ids(output_path: Path) -> set:
    if not RESUME or not output_path.exists():
        return set()
    ids = set()
    for ex in iter_jsonl(output_path):
        if ex.get("id"):
            ids.add(ex["id"])
    return ids


def enrich_jsonl_file(input_path: Path, output_path: Path, max_records: Optional[int] = None) -> pd.DataFrame:
    if not input_path.exists():
        raise FileNotFoundError(f"Input file does not exist: {input_path}")

    total = count_jsonl(input_path)
    processed_ids = load_processed_ids(output_path)
    if processed_ids:
        print(f"Resume enabled: {len(processed_ids)} ids already in {output_path}")
    else:
        if output_path.exists() and not RESUME:
            print(f"RESUME=False: overwriting {output_path}")
            output_path.unlink()

    stats = Counter()
    rows_for_stats = []
    n_seen = 0
    n_selected = 0

    pbar = tqdm(iter_jsonl(input_path), total=total, desc=f"enrich {input_path.name}")
    for ex in pbar:
        n_seen += 1
        ex_id = ex.get("id")

        if not record_passes_filters(ex):
            # Если фильтр стоит на cinema, остальные записи всё равно сохраняем неизменёнными,
            # чтобы output был полноценной копией input.
            if ex_id not in processed_ids:
                ex2 = dict(ex)
                ex2.setdefault("gold_enrichment_status", "not_selected_by_filter")
                append_jsonl(output_path, ex2, fsync=FSYNC_EACH_LINE)
            stats["not_selected_by_filter"] += 1
            continue

        n_selected += 1
        if max_records is not None and n_selected > max_records:
            break

        if ex_id in processed_ids:
            stats["resume_skipped"] += 1
            continue

        t0 = time.time()
        try:
            enriched = enrich_one_example(ex)
        except KeyboardInterrupt:
            raise
        except Exception as e:
            enriched = dict(ex)
            enriched["gold_enrichment_status"] = "fatal_error"
            enriched["gold_enrichment_error"] = repr(e)
            enriched["gold_enrichment_traceback"] = traceback.format_exc()[-3000:]

        dt = time.time() - t0
        status = enriched.get("gold_enrichment_status", "unknown")
        stats[status] += 1

        rows_for_stats.append({
            "id": enriched.get("id"),
            "domain": enriched.get("domain"),
            "complexity": enriched.get("complexity"),
            "status": status,
            "method": enriched.get("gold_enrichment_method"),
            "old": enriched.get("gold_old_count", safe_len_list(ex.get("gold_answer_qids"))),
            "full": enriched.get("gold_full_count"),
            "added": enriched.get("gold_added_count"),
            "seconds": round(dt, 3),
        })

        append_jsonl(output_path, enriched, fsync=FSYNC_EACH_LINE)
        pbar.set_postfix({
            "status": status,
            "old": rows_for_stats[-1]["old"],
            "full": rows_for_stats[-1]["full"],
            "ok": stats["ok"] + stats["ok_but_original_not_subset"],
            "too_many": stats["too_many_candidates"],
            "err": stats["error"] + stats["fatal_error"],
        })

    print("\nDone:", output_path)
    print(dict(stats))
    return pd.DataFrame(rows_for_stats)


if RUN_ENRICHMENT_NOW:
    stats_frames = []
    if ENRICH_MAIN:
        stats_main = enrich_jsonl_file(INPUT_MAIN_PATH, OUTPUT_MAIN_PATH, max_records=MAX_RECORDS)
        stats_main["split"] = "main"
        stats_frames.append(stats_main)
    if ENRICH_ZERO:
        stats_zero = enrich_jsonl_file(INPUT_ZERO_PATH, OUTPUT_ZERO_PATH, max_records=MAX_RECORDS)
        stats_zero["split"] = "zero"
        stats_frames.append(stats_zero)
    enrichment_stats_df = pd.concat(stats_frames, ignore_index=True) if stats_frames else pd.DataFrame()
    display(enrichment_stats_df.tail(20))
else:
    print("RUN_ENRICHMENT_NOW=False; enrichment cell skipped.")

Resume enabled: 1034 ids already in out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl


enrich dataset_main.jsonl: 100%|██████████| 1094/1094 [09:58<00:00,  1.83it/s, status=ok, old=1, full=1, ok=11, too_many=0, err=3]     


Done: out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl
{'resume_skipped': 1080, 'error': 3, 'ok': 11}


,id,domain,complexity,status,method,old,full,added,seconds,split
0,museums_l5_1144,museums,L5,error,None,7,NaN,NaN,80.227,main
1,paintings_l5_1081,paintings,L5,error,None,1,NaN,NaN,102.861,main
2,paintings_l5_1083,paintings,L5,ok,wrapped_original,2,2.0,0.0,64.392,main
3,paintings_l5_1085,paintings,L5,error,None,1,NaN,NaN,121.601,main
4,paintings_l5_1087,paintings,L5,ok,fast_qid_rewrite,1,1.0,0.0,4.014,main
5,paintings_l5_1089,paintings,L5,ok,fast_qid_rewrite,1,2.0,1.0,3.441,main
6,paintings_l5_1093,paintings,L5,ok,fast_qid_rewrite,1,4.0,3.0,22.328,main
7,paintings_l5_1095,paintings,L5,ok,fast_qid_rewrite,1,1.0,0.0,1.540,main
8,paintings_l5_1097,paintings,L5,ok,fast_qid_rewrite,1,4.0,3.0,2.316,main
9,paintings_l4_1080,paintings,L4,ok,fast_qid_rewrite,1,1.0,0.0,1.974,main


In [36]:
# ============================================================
# 8. Post-checks and diagnostics
# ============================================================

def load_enriched_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"No file: {path}")
        return pd.DataFrame()
    rows = read_jsonl(path)
    df = pd.DataFrame(rows)
    for col in [
        "id", "domain", "complexity", "gold_enrichment_status", "gold_old_count",
        "gold_full_count", "gold_added_count", "gold_missing_from_original_count",
        "gold_answer_qids", "gold_answer_qids_original", "gold_answer_qids_full"
    ]:
        if col not in df.columns:
            df[col] = None
    df["gold_old_count_calc"] = df["gold_answer_qids_original"].apply(safe_len_list)
    df["gold_current_count_calc"] = df["gold_answer_qids"].apply(safe_len_list)
    df["gold_full_count_calc"] = df["gold_answer_qids_full"].apply(safe_len_list)
    return df


def summarize_enriched(path: Path) -> pd.DataFrame:
    df = load_enriched_df(path)
    if df.empty:
        return df

    print("Loaded:", len(df), path)
    print("\nStatus counts:")
    display(df["gold_enrichment_status"].value_counts(dropna=False).to_frame("n"))

    ok = df[df["gold_enrichment_status"].isin(list(SUCCESS_STATUSES))].copy()
    if len(ok):
        print("\nEnrichment gain by domain/complexity:")
        display(
            ok.groupby(["domain", "complexity"], dropna=False)
              .agg(
                  n=("id", "count"),
                  old_mean=("gold_old_count", "mean"),
                  full_mean=("gold_full_count", "mean"),
                  added_mean=("gold_added_count", "mean"),
                  old_median=("gold_old_count", "median"),
                  full_median=("gold_full_count", "median"),
                  max_full=("gold_full_count", "max"),
              )
              .reset_index()
        )

        print("\nRows with biggest added gold:")
        show_cols = ["id", "domain", "complexity", "query_text_ru", "gold_old_count", "gold_full_count", "gold_added_count"]
        display(ok.sort_values("gold_added_count", ascending=False)[show_cols].head(20))

        suspicious = ok[ok["gold_missing_from_original_count"].fillna(0).astype(float) > 0]
        if len(suspicious):
            print("\n[WARN] Old gold not subset of new full gold. Check these rows:")
            display(suspicious[["id", "domain", "complexity", "gold_missing_from_original_count", "gold_enrichment_method"]].head(20))

    too_many = df[df["gold_enrichment_status"] == "too_many_candidates"]
    if len(too_many):
        print("\nToo wide queries, not enriched as partial:")
        display(too_many[["id", "domain", "complexity", "query_text_ru", "gold_old_count"]].head(20))

    csv_path = path.with_suffix(".diagnostics.csv")
    cols = [c for c in ["id", "domain", "complexity", "gold_enrichment_status", "gold_enrichment_method", "gold_old_count", "gold_full_count", "gold_added_count", "gold_missing_from_original_count", "query_text_ru"] if c in df.columns]
    df[cols].to_csv(csv_path, index=False)
    print("Diagnostics CSV:", csv_path)
    return df

if OUTPUT_MAIN_PATH.exists():
    df_enriched_main = summarize_enriched(OUTPUT_MAIN_PATH)

Loaded: 1087 out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl

Status counts:


,n
gold_enrichment_status,
ok,824
error,116
skip_cannot_infer_answer_var,69
ok_but_original_not_subset,43
too_many_candidates,34
skip_no_sparql,1



Enrichment gain by domain/complexity:


,domain,complexity,n,old_mean,full_mean,added_mean,old_median,full_median,max_full
0,airports,L1,10,11.800000,237.500000,225.700000,12.0,155.0,667.0
1,airports,L2,11,12.000000,255.727273,243.727273,12.0,179.0,653.0
2,airports,L3,10,11.800000,24.300000,12.500000,13.5,14.5,58.0
3,airports,L4,8,8.375000,8.750000,0.375000,6.5,6.5,27.0
4,cars,L1,2,12.000000,493.000000,481.000000,12.0,493.0,768.0
...,...,...,...,...,...,...,...,...,...
77,universities,L4,9,20.000000,119.777778,99.777778,24.0,39.0,762.0
78,videogames,L1,20,35.250000,103.150000,67.900000,14.0,42.5,422.0
79,videogames,L2,19,16.789474,53.421053,36.631579,12.0,15.0,592.0
80,videogames,L3,16,7.125000,10.187500,3.062500,5.0,5.0,42.0



Rows with biggest added gold:


...


[WARN] Old gold not subset of new full gold. Check these rows:


,id,domain,complexity,gold_missing_from_original_count,gold_enrichment_method
1,cinema_l1_0002,cinema,L1,1.0,wrapped_original
6,cinema_l1_0012,cinema,L1,10.0,wrapped_original
21,cinema_l1_0042,cinema,L1,1.0,wrapped_original
24,cinema_l1_0048,cinema,L1,2.0,wrapped_original
65,cinema_l2_0076,cinema,L2,1.0,wrapped_original
68,cinema_l2_0082,cinema,L2,1.0,wrapped_original
94,cinema_l5_0084,cinema,L5,3.0,wrapped_original
108,cinema_l5_0092,cinema,L5,3.0,wrapped_original
154,countries_l1_00156,countries,L1,1.0,wrapped_original
183,countries_l3_00189,countries,L3,1.0,wrapped_original



Too wide queries, not enriched as partial:


,id,domain,complexity,query_text_ru,gold_old_count
121,cinema_l1_0126,cinema,L1,"Назови 5 фильмов, жанра «документальный фильм»...",8
371,sports_clubs_l1_0156,sports_clubs,L1,"Назови 5 спортивных клубов, по виду спорта: фу...",173
372,sports_clubs_l1_0158,sports_clubs,L1,"Назови 5 спортивных клубов, по виду спорта: фу...",44
382,sports_clubs_l1_0178,sports_clubs,L1,"Назови 5 спортивных клубов, по виду спорта: фу...",6
401,museums_l1_0159,museums,L1,"Назови 5 известных музеев, которые находятся в...",43
402,museums_l1_0161,museums,L1,"Назови 5 известных музеев, которые находятся в...",110
404,museums_l1_0165,museums,L1,"Назови 5 известных музеев, которые находятся в...",18
406,museums_l1_0169,museums,L1,"Назови 5 известных музеев, которые находятся в...",27
408,people_l1_0153,people,L1,"Назови 5 известных персон, профессии: «актёр»,...",250
409,people_l1_0155,people,L1,"Назови 5 известных персон, профессии: «писател...",250


Diagnostics CSV: out_wikidata_benchmark/dataset_main.full_gold_enriched.diagnostics.csv


In [30]:
%pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp310-cp310-macosx_11_0_arm64.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jinja2]

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
# ============================================================
# Final statistics for ready Wikidata dataset
# ============================================================

import json
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

# Финальный датасет после обновления gold'ов
DATASET_PATH = Path("out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl")

rows = []
bad_lines = []

with DATASET_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except Exception as e:
            bad_lines.append((line_no, str(e), line[:300]))

print("Dataset:", DATASET_PATH)
print("Valid rows:", len(rows))
print("Bad JSONL lines:", len(bad_lines))

if bad_lines:
    print("\nBad lines preview:")
    for x in bad_lines[:5]:
        print(x)

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# Helper columns
# ------------------------------------------------------------

def get_gold_count(row):
    """
    Считаем датасет уже готовым.
    Основной источник истины — gold_answer_qids.
    Если его нет, fallback на gold_answer_qids_full / gold_full_count.
    """
    qids = row.get("gold_answer_qids")
    if isinstance(qids, list):
        return len(qids)

    full_qids = row.get("gold_answer_qids_full")
    if isinstance(full_qids, list):
        return len(full_qids)

    if pd.notna(row.get("gold_full_count")):
        return int(row.get("gold_full_count"))

    return 0


df["gold_count"] = df.apply(get_gold_count, axis=1)

complexity_order = ["L1", "L2", "L3", "L4", "L5"]
df["complexity"] = pd.Categorical(
    df["complexity"],
    categories=complexity_order,
    ordered=True,
)

# ------------------------------------------------------------
# 1. Overall summary
# ------------------------------------------------------------

overall = pd.DataFrame([{
    "Всего запросов": len(df),
    "Количество доменов": df["domain"].nunique(),
    "Среднее число gold-ответов": df["gold_count"].mean(),
    "Медианное число gold-ответов": df["gold_count"].median(),
    "Минимум gold-ответов": df["gold_count"].min(),
    "Максимум gold-ответов": df["gold_count"].max(),
    "Всего gold-связей": df["gold_count"].sum(),
}])

display(Markdown("## Общая статистика датасета"))
display(
    overall.style.format({
        "Среднее число gold-ответов": "{:.2f}",
        "Медианное число gold-ответов": "{:.2f}",
    })
)

# ------------------------------------------------------------
# 2. Statistics by domain
# ------------------------------------------------------------

domain_stats = (
    df.groupby("domain", observed=True)
      .agg(
          Запросов=("id", "count"),
          Среднее_gold=("gold_count", "mean"),
          Медиана_gold=("gold_count", "median"),
          Минимум_gold=("gold_count", "min"),
          Максимум_gold=("gold_count", "max"),
          Всего_gold_связей=("gold_count", "sum"),
      )
      .reset_index()
      .rename(columns={"domain": "Домен"})
      .sort_values(["Запросов", "Домен"], ascending=[False, True])
)

display(Markdown("## Статистика по доменам"))
display(
    domain_stats.style.format({
        "Среднее_gold": "{:.2f}",
        "Медиана_gold": "{:.2f}",
    })
)

# ------------------------------------------------------------
# 3. Statistics by complexity level
# ------------------------------------------------------------

complexity_stats = (
    df.groupby("complexity", observed=True)
      .agg(
          Запросов=("id", "count"),
          Среднее_gold=("gold_count", "mean"),
          Медиана_gold=("gold_count", "median"),
          Минимум_gold=("gold_count", "min"),
          Максимум_gold=("gold_count", "max"),
          Всего_gold_связей=("gold_count", "sum"),
      )
      .reset_index()
      .rename(columns={"complexity": "Сложность"})
)

display(Markdown("## Статистика по уровням сложности"))
display(
    complexity_stats.style.format({
        "Среднее_gold": "{:.2f}",
        "Медиана_gold": "{:.2f}",
    })
)

# ------------------------------------------------------------
# 4. Domain x complexity: number of queries
# ------------------------------------------------------------

domain_complexity_count = (
    df.pivot_table(
        index="domain",
        columns="complexity",
        values="id",
        aggfunc="count",
        fill_value=0,
        observed=True,
    )
    .reset_index()
    .rename(columns={"domain": "Домен"})
)

existing_complexities = [
    c for c in complexity_order
    if c in domain_complexity_count.columns
]

domain_complexity_count["Итого"] = domain_complexity_count[existing_complexities].sum(axis=1)

domain_complexity_count = domain_complexity_count.sort_values(
    ["Итого", "Домен"],
    ascending=[False, True],
)

display(Markdown("## Количество запросов по доменам и уровням сложности"))
display(domain_complexity_count)

# ------------------------------------------------------------
# 5. Domain x complexity: average number of gold answers
# ------------------------------------------------------------

domain_complexity_avg_gold = (
    df.pivot_table(
        index="domain",
        columns="complexity",
        values="gold_count",
        aggfunc="mean",
        fill_value=0,
        observed=True,
    )
    .reset_index()
    .rename(columns={"domain": "Домен"})
)

display(Markdown("## Среднее число gold-ответов по доменам и уровням сложности"))
display(
    domain_complexity_avg_gold.style.format({
        c: "{:.2f}" for c in complexity_order
        if c in domain_complexity_avg_gold.columns
    })
)

# ------------------------------------------------------------
# 6. Save statistics to CSV
# ------------------------------------------------------------

STATS_DIR = Path("out_wikidata_benchmark/final_stats")
STATS_DIR.mkdir(parents=True, exist_ok=True)

overall.to_csv(STATS_DIR / "overall_stats.csv", index=False)
domain_stats.to_csv(STATS_DIR / "domain_stats.csv", index=False)
complexity_stats.to_csv(STATS_DIR / "complexity_stats.csv", index=False)
domain_complexity_count.to_csv(STATS_DIR / "domain_complexity_count.csv", index=False)
domain_complexity_avg_gold.to_csv(STATS_DIR / "domain_complexity_avg_gold.csv", index=False)

print("\nSaved CSV stats to:", STATS_DIR)

Dataset: out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl
Valid rows: 1087
Bad JSONL lines: 0


## Общая статистика датасета

,Всего запросов,Количество доменов,Среднее число gold-ответов,Медианное число gold-ответов,Минимум gold-ответов,Максимум gold-ответов,Всего gold-связей
0,1087,19,54.07,11.00,0,768,58769


## Статистика по доменам

,Домен,Запросов,Среднее_gold,Медиана_gold,Минимум_gold,Максимум_gold,Всего_gold_связей
2,cinema,191,48.25,9.00,2,618,9215
4,dishes,103,22.41,8.00,1,156,2308
3,countries,73,7.40,4.00,1,60,540
9,music_albums,73,54.53,14.00,4,480,3981
11,people,67,102.18,34.00,1,591,6846
18,videogames,65,50.42,10.00,1,592,3277
10,paintings,61,54.49,8.00,1,587,3324
8,museums,47,54.02,17.00,1,324,2539
12,physicists,47,81.19,15.00,1,663,3816
1,cars,45,45.49,14.00,5,768,2047


...

## Количество запросов по доменам и уровням сложности

complexity,Домен,L1,L2,L3,L4,L5,Итого
2,cinema,41,30,20,66,34,191
4,dishes,16,21,31,20,15,103
3,countries,10,26,25,2,10,73
9,music_albums,15,16,25,15,2,73
11,people,15,17,25,10,0,67
18,videogames,21,19,16,9,0,65
10,paintings,10,12,17,11,11,61
8,museums,10,7,10,10,10,47
12,physicists,10,12,15,10,0,47
1,cars,8,12,15,10,0,45


## Среднее число gold-ответов по доменам и уровням сложности

complexity,Домен,L1,L2,L3,L4,L5
0,airports,199.92,235.42,24.30,8.75,0.00
1,cars,131.88,21.17,11.93,55.90,0.00
2,cinema,113.32,8.13,162.05,10.42,11.65
3,countries,18.80,4.08,9.00,2.00,1.70
4,dishes,19.69,30.67,35.94,10.80,1.27
5,geo,3.60,2.11,2.57,2.40,0.00
6,geo_ru,6.80,25.33,68.67,2.75,195.00
7,mathematicians,322.00,22.25,26.53,5.40,0.00
8,museums,71.00,123.00,77.00,15.00,4.80
9,music_albums,99.67,22.00,73.72,18.33,8.00



Saved CSV stats to: out_wikidata_benchmark/final_stats
